# Projet : Stage

## 1. Hiérarchisation des données

### 1.1. Arrondissement/Cantons/Communes/Departements/Regions

In [1]:
import json

In [36]:
def fillChamp(dicChamps, dic_hierarchisee, rang) :

    for i in range(len(dicChamps['features'])) :
        champ = dicChamps['features'][i]['properties']['nom']
        if champ not in dic_hierarchisee[rang] :
            dic_hierarchisee[rang].append(champ.lower())
        
    return 

def fillDictionnaire() :
    fichiers = ['arrondissements', 'cantons', 'communes', 'departements', 'regions']
    dic_hierarchisee = {}

    for fichier in fichiers :
        dic_hierarchisee[fichier] = []
        mon_json = open(f"Education/levels/france-geojson/{fichier}-avec-outre-mer.geojson")
        data = json.load(mon_json)
        mon_json.close()

        fillChamp(data, dic_hierarchisee, fichier)

    return dic_hierarchisee

In [37]:
dic_hierarchisee = fillDictionnaire()

### 1.2 Quartiers

In [38]:
import csv

In [39]:

fichier = open("Education/levels/liste-correspondance-qp2024-qp2015.csv", "r", encoding="utf-8")
reader = csv.reader(fichier, delimiter=";")
listeQuartiers = list(reader)[1:]

In [40]:
dic_hierarchisee['quartiers'] = []
for i in range(len(listeQuartiers)) :
    quartier = listeQuartiers[i][1]
    if quartier not in dic_hierarchisee['quartiers'] :
        dic_hierarchisee['quartiers'].append(quartier.lower())

dic_hierarchisee['QP'] = []
for i in range(len(listeQuartiers)) :
    quartier = listeQuartiers[i][3]
    if quartier not in dic_hierarchisee['QP'] :
        dic_hierarchisee['QP'].append(quartier.lower())

del i, listeQuartiers, quartier, fichier, reader

In [41]:
for key in dic_hierarchisee:
    dic_hierarchisee[key].sort()

## 2. Identification des attributs spatiaux dans un fichier csv/xlsx

### 2.1 Algorithme de recherche

Mon objectif est de parcourir un tableau en vérifiant à chaque cellule si elle appartient à une donnée de mon dictionnaire jusqu'à trouver la plus petite et la plus grande granularité

In [42]:
# Tableau rangé par granularité des champs
champs = ['QP', 'quartiers', 'arrondissements', 'cantons', 'communes', 'departements', 'regions']
dic_hierarchisee = {champ: dic_hierarchisee[champ] for champ in champs if champ in dic_hierarchisee}

### 2.2 Fichiers csv & xlsx

In [ ]:
import os
import pandas as pd

def getFiles(origine):
    fichiers = []
    for dossier in os.walk(origine):
        for fichier in dossier[2]:
            if fichier.endswith('.csv') or fichier.endswith('.xlsx'):
                fichiers.append(os.path.join(dossier[0], fichier))
    return fichiers

documents = getFiles("Education")

['Education/excel/EDUC_2023_V1.xlsx', 'Education/excel/pop-16ans-dipl6820_v2.xlsx', 'Education/excel/base-ic-diplomes-formation-2020.xlsx', 'Education/csv/annuaire-de-leducation.csv', 'Education/csv/fr-esr-insersup.csv', 'Education/csv/formations.csv', 'Education/csv/fr-en-baccalaureat-par-academie.csv', 'Education/csv/lycees-donnees-generales.csv', 'Education/csv/fr-esr-atlas_regional-effectifs-d-etudiants-inscrits_agregeables.csv', 'Education/csv/fr-en-adresse-et-geolocalisation-etablissements-premier-et-second-degre.csv']


In [ ]:
metadonnees = {}

def rechercheAttributs(listeChamps, attributs_trouves) :
    for i in range(10) :
            for j in range (len(listeChamps[0]) - 1) :
                for champ, valeur in dic_hierarchisee.items() :
                    if isinstance(listeChamps[i][j], str) :
                        listeChamps[i][j] = listeChamps[i][j].lower()
                    if listeChamps[i][j] in valeur :
                        if champ not in attributs_trouves :
                            attributs_trouves.append(champ)

for document in documents :

    if document.endswith('.csv') :
        fichier = open(document, encoding="utf-8")
        reader = csv.reader(fichier, delimiter=";")
        listeChamps = list(reader)[1:]
        fichier.close()

        attributs_trouves = []
        rechercheAttributs(listeChamps, attributs_trouves)

        if len(attributs_trouves) > 0 :
            attributs_trouves.pop(-1)
            
            granularite_basse = next((champ for champ in champs if champ in attributs_trouves), None)
            granularite_haute = next((champ for champ in reversed(champs) if champ in attributs_trouves), None)

        else :
            fichier = open(document, encoding="utf-8")
            reader = csv.reader(fichier, delimiter=",")
            fichier.close()
            rechercheAttributs(listeChamps, attributs_trouves)

    elif document.endswith('.xlsx') :
        
        df = pd.read_excel(document, engine='openpyxl')
        listeChamps = df.values.tolist()
        attributs_trouves = []
        print(listeChamps)

        if len(attributs_trouves) > 0 :
            attributs_trouves.pop(-1)
            
            granularite_basse = next((champ for champ in champs if champ in attributs_trouves), None)
            granularite_haute = next((champ for champ in reversed(champs) if champ in attributs_trouves), None)

    metadonnees[document] = {
        'granularite_basse': granularite_basse,
        'granularite_haute': granularite_haute
    }


IndexError: list index out of range